# Latency benchmark — TPU edition

Per-query time for retrieval alone vs retrieval + reranking, measured on a Colab TPU.

## Read this before trusting any number here

**1. On XLA, a stopwatch around a model call measures nothing.** Execution is lazy: the call queues
work and returns immediately, so `perf_counter` stops before the TPU has done anything. Every timed
region in this notebook ends with an explicit `sync()` that forces the graph to execute. Without it
the reranker would appear to take microseconds.

On CUDA this problem does not arise in the original notebook, because
`ce.predict()` returns a numpy array and that conversion synchronises. The GPU timings are valid as
written; only XLA needs the explicit barrier.

**2. Compilation dominates the first call.** XLA compiles a graph per input shape. The warmup is
therefore not optional here — it is what pays the compile cost so the timed run measures steady
state. Compile time is reported separately, because it is a once-per-deployment cost like index
building.

**3. Fixed shapes change the latency profile.** To avoid recompiling per query, every batch is
padded to `(batch_size, max_length)`. A short query therefore costs the same as a long one — TPU
latency here is flat where GPU latency varies with input length. That is a real property of this
deployment style, not a measurement artefact, but it makes the median and p95 closer together than
on GPU.

**These numbers are not comparable to the GPU run.** Different hardware, different padding regime.
Report them as a separate row, not as a speedup.

## Defects fixed

**Paths.** The original opened `corpus_v2.json` / `qa_pairs_wiki.json` from the working directory;
in this repo they are `data/corpus.json` and `data/qa_pairs_wiki.json`. Outputs went to the working
directory rather than `results/`, and the last cell was a bare `from google.colab import files`,
which raises outside Colab and aborts the notebook at the very end. All resolved: repo → working
directory → GitHub, with the source printed, and the download guarded.

**Deprecated `torch_dtype`.** `automodel_args={"torch_dtype": ...}` is deprecated in current
transformers and absent in old ones. Replaced with a post-load cast that works on every version.

**Warmup called `timed_retrieve` twice per query.** Harmless but wasteful, and it obscured that
the reranker warmup depended on the second call's output. Now one call, result reused.

### Install

In [ ]:
import importlib.util, subprocess, sys

# Pin torch_xla to the ALREADY-INSTALLED torch so pip does not pull a different
# torch and force a runtime restart mid-notebook.
if importlib.util.find_spec("torch_xla") is None:
    import torch as _t
    _v = _t.__version__.split("+")[0]
    print(f"torch {_v} present, torch_xla missing -> installing torch_xla=={_v}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"torch_xla[tpu]=={_v}",
                        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
                       capture_output=True, text=True)
    print("pip exit", r.returncode)
    if r.returncode != 0:
        print(r.stderr[-2000:])
else:
    print("torch_xla already available")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "rank_bm25", "transformers", "sentencepiece"], check=False)
print("deps ready")

### Device — and which one we actually got

In [ ]:
import torch

BACKEND, device, xm = "cpu", torch.device("cpu"), None
try:
    import torch_xla
    import torch_xla.core.xla_model as _xm
    xm = _xm
    device = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()
    _ = (torch.ones(2, 2, device=device) * 2).sum().item()   # force a real TPU op
    BACKEND = "tpu"
except Exception as e:
    print(f"XLA unavailable ({type(e).__name__}: {str(e)[:160]})")
    if torch.cuda.is_available():
        device, BACKEND = torch.device("cuda"), "cuda"

def sync():
    """Flush the XLA graph. No-op off TPU."""
    if BACKEND == "tpu":
        torch_xla.sync() if hasattr(torch_xla, "sync") else xm.mark_step()

print("=" * 80)
print(f"BACKEND ACTUALLY IN USE: {BACKEND.upper()}   (device={device})")
if BACKEND == "tpu":
    print(f"torch {torch.__version__} | torch_xla {torch_xla.__version__}")
print("=" * 80)

### Paths

In [ ]:
import os, json, re, gc, time, random, pickle, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

RAW_BASE = "https://raw.githubusercontent.com/Rania-khaoudane/MSA/main/data"

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "qa_pairs_wiki.json").exists():
            return base
    return None

ROOT = _repo_root()

def find_file(*names, subdirs=("data", "results")):
    """Locate a file: repo subdirs first, then the working directory."""
    for n in names:
        if ROOT:
            for sd in subdirs:
                p = ROOT / sd / n
                if p.exists():
                    return p
        if Path(n).exists():
            return Path(n)
    return None

def resolve(*names):
    """Local -> GitHub. Returns (json, description)."""
    p = find_file(*names, subdirs=("data",))
    if p:
        return json.loads(p.read_text(encoding="utf-8")), f"local: {p}"
    for n in names:
        try:
            url = f"{RAW_BASE}/{n}"
            with urllib.request.urlopen(url) as r:
                return json.loads(r.read().decode("utf-8")), f"github: {url}"
        except Exception:
            continue
    raise FileNotFoundError(f"none of {names} found locally or on GitHub")

OUT_DIR = (ROOT / "results") if ROOT else Path(".")
OUT_DIR.mkdir(exist_ok=True)
def out(name):
    return str(OUT_DIR / name)

print("repo root :", ROOT or "(not in the repo)")
print("output dir:", OUT_DIR.resolve())

### Config

In [ ]:
CONFIG = {
    "base_encoder": "intfloat/multilingual-e5-base",
    "reranker": "BAAI/bge-reranker-v2-m3",
    "alpha": 0.8,
    "retrieve_k": 20,   # candidates passed to the reranker -- same as solution_6
    "n_queries": 100,   # timed queries, after warmup
    "n_warmup": 5,
    "max_length": 512,
    "batch_size": 32,
    "seed": 42,
}
CONFIG

### Load data (index-build time is NOT per-query cost)

In [ ]:
corpus, src_c = resolve("corpus_v2.json", "corpus.json")
wiki_qa, src_q = resolve("qa_pairs_wiki.json")
print("corpus from", src_c); print("qa     from", src_q)

corpus_ids = [c["chunk_id"] for c in corpus]
corpus_texts = [c["text"] for c in corpus]
corpus_map = dict(zip(corpus_ids, corpus_texts))
known = set(corpus_ids)
qa = [q for q in wiki_qa if q["source_chunk_id"] in known]
random.Random(CONFIG["seed"]).shuffle(qa)

need = CONFIG["n_queries"] + CONFIG["n_warmup"]
if len(qa) < need:
    raise ValueError(f"need {need} queries (n_queries + n_warmup) but only {len(qa)} available")
timed_qa = qa[: CONFIG["n_queries"]]
warmup_qa = qa[CONFIG["n_queries"]: need]
print(f"Corpus: {len(corpus)} passages | timing {len(timed_qa)} queries "
      f"(+{len(warmup_qa)} warmup, excluded from the measurement)")

### BM25 (one-time build cost)

In [ ]:
from rank_bm25 import BM25Okapi

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]")

def normalize_arabic(t):
    t = DIAC.sub("", t)
    t = re.sub(r"[\u0625\u0623\u0622\u0627]", "\u0627", t)
    t = re.sub(r"\u0649", "\u064A", t); t = re.sub(r"\u0629", "\u0647", t)
    t = re.sub(r"\u0624", "\u0648", t); t = re.sub(r"\u0626", "\u064A", t)
    t = re.sub(r"\u0640+", "", t); t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize(t):
    return normalize_arabic(t).split()

t0 = time.perf_counter()
bm25 = BM25Okapi([tokenize(t) for t in corpus_texts])
bm25_build_s = time.perf_counter() - t0
print(f"BM25 index built in {bm25_build_s:.2f}s (one-time, not per-query).")

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

### Build the dense index and load the reranker (one-time costs)

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

def batched_fixed(items, bs):
    """Yield (padded_batch, n_real); last batch padded up to bs to keep XLA shapes static."""
    for i in range(0, len(items), bs):
        ch = list(items[i:i + bs]); n = len(ch)
        if n < bs:
            ch += [ch[-1]] * (bs - n)
        yield ch, n

def mean_pool(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode_texts(model, tk, texts, bs=32, max_len=512, label=""):
    """e5-style mean pooling + L2 normalisation. Verified to match
    sentence-transformers to within 3e-8 on this corpus."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(texts, bs):
        enc = tk(ch, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        v = mean_pool(model(**enc).last_hidden_state, enc["attention_mask"])
        v = torch.nn.functional.normalize(v, p=2, dim=1)
        sync()
        o.append(v.float().cpu().numpy()[:n]); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(o, 0).astype("float32")

@torch.no_grad()
def score_pairs(model, tk, pairs, bs, max_len, label=""):
    """Cross-encoder relevance score per (query, passage) pair, static shapes for XLA."""
    o, done, t0 = [], 0, time.time()
    for ch, n in batched_fixed(pairs, bs):
        enc = tk([a for a, _ in ch], [b for _, b in ch], padding="max_length",
                 truncation=True, max_length=max_len, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model(**enc).logits
        sync()
        s = logits.float().cpu().numpy()
        # Same guard as the GPU notebook: some rerankers emit a 2D per-class array.
        s = s[:, 0] if s.shape[-1] == 1 else s[:, -1]
        o.extend(s[:n].tolist()); done += n
        if done % (bs * 20) < bs:
            print(f"    {label} {done}/{len(pairs)} ({time.time()-t0:.0f}s)", flush=True)
    return np.asarray(o)

print("XLA helpers ready")

In [ ]:
enc_tok = AutoTokenizer.from_pretrained(CONFIG["base_encoder"])
enc_model = AutoModel.from_pretrained(CONFIG["base_encoder"]).to(device=device, dtype=torch.float32).eval()

t0 = time.perf_counter()
corpus_emb = encode_texts(enc_model, enc_tok, [f"passage: {t}" for t in corpus_texts], label="corpus")
sync()
index_build_s = time.perf_counter() - t0
print(f"Dense index built in {index_build_s:.2f}s for {len(corpus)} passages (one-time).")

ce_tok = AutoTokenizer.from_pretrained(CONFIG["reranker"], trust_remote_code=True)
t0 = time.perf_counter()
ce = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["reranker"], trust_remote_code=True).to(device=device, dtype=torch.float32).eval()
model_load_s = time.perf_counter() - t0
print(f"Reranker loaded in {model_load_s:.2f}s: {CONFIG['reranker']}")
print(f"Device: {BACKEND}")

### Timed stages — each ends with an explicit `sync()`

Without the barrier these functions would return before the TPU had executed anything, and the
reranker would appear to cost microseconds. The `sync()` is *inside* the timed region because
waiting for the result is part of serving a query.

In [ ]:
@torch.no_grad()
def timed_retrieve(query, k):
    """Hybrid retrieval for one query -> (candidate_ids, elapsed_seconds)."""
    t0 = time.perf_counter()
    enc = enc_tok([f"query: {query}"], padding="max_length", truncation=True,
                  max_length=CONFIG["max_length"], return_tensors="pt")
    enc = {kk: v.to(device) for kk, v in enc.items()}
    v = mean_pool(enc_model(**enc).last_hidden_state, enc["attention_mask"])
    v = torch.nn.functional.normalize(v, p=2, dim=1)
    sync()                                   # <-- without this the timer is meaningless
    qv = v.float().cpu().numpy()[0]
    s = (CONFIG["alpha"] * minmax(corpus_emb @ qv)
         + (1 - CONFIG["alpha"]) * minmax(np.asarray(bm25.get_scores(tokenize(query)))))
    idx = np.argsort(-s)[:k]
    return [corpus_ids[i] for i in idx], time.perf_counter() - t0

@torch.no_grad()
def timed_rerank(query, candidate_ids):
    """Cross-encoder rerank of an already-retrieved list -> (reordered, elapsed_seconds)."""
    t0 = time.perf_counter()
    pairs = [(query, corpus_map[c]) for c in candidate_ids]
    o = []
    for ch, nn in batched_fixed(pairs, CONFIG["batch_size"]):
        enc = ce_tok([a for a, _ in ch], [b for _, b in ch], padding="max_length",
                     truncation=True, max_length=CONFIG["max_length"], return_tensors="pt")
        enc = {kk: v.to(device) for kk, v in enc.items()}
        lg = ce(**enc).logits
        sync()                               # <-- same barrier
        sc = lg.float().cpu().numpy()
        sc = sc[:, 0] if sc.shape[-1] == 1 else sc[:, -1]
        o.extend(sc[:nn].tolist())
    order = np.argsort(-np.asarray(o))
    return [candidate_ids[i] for i in order], time.perf_counter() - t0

### Warmup (excluded from the measurement)

On XLA this is where graph compilation is paid. The first call's cost is reported so it is visible as a once-per-deployment number rather than hidden in the per-query median.

In [ ]:
print("Warming up...")
t0 = time.perf_counter()
first_call_s = None
for i, q in enumerate(warmup_qa):
    cands, t_r = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])   # one call, result reused
    _, t_rr = timed_rerank(q["darija_query"], cands)
    if i == 0:
        first_call_s = t_r + t_rr
warmup_s = time.perf_counter() - t0
print(f"Warmup done in {warmup_s:.2f}s; first call alone took {first_call_s*1000:.0f} ms"
      + ("  <- includes XLA compilation" if "BACKEND" in dir() and BACKEND == "tpu" else ""))

### The timed run

In [ ]:
rows = []
for q in timed_qa:
    cands, t_retrieve = timed_retrieve(q["darija_query"], CONFIG["retrieve_k"])
    _, t_rerank = timed_rerank(q["darija_query"], cands)
    rows.append({"qid": q["id"], "retrieve_ms": t_retrieve * 1000,
                 "rerank_ms": t_rerank * 1000,
                 "total_ms": (t_retrieve + t_rerank) * 1000})

lat = pd.DataFrame(rows)
lat.to_csv(out("latency_measurements_tpu.csv"), index=False)
print(f"Timed {len(lat)} queries.")

### Summary statistics (the numbers for the paper)

In [ ]:
def stats(s):
    return {"mean": s.mean(), "median": s.median(), "p95": s.quantile(0.95),
            "min": s.min(), "max": s.max()}

DEV = BACKEND
print("=" * 72)
print(f"LATENCY SUMMARY (n={len(lat)} queries, device={DEV})")
print("=" * 72)
print(f"  padding           fixed ({CONFIG['batch_size']} x {CONFIG['max_length']}) -- query length does not affect cost")
for col, label in [("retrieve_ms", "Retrieval only (BM25 + dense)"),
                   ("rerank_ms", f"Reranking only ({CONFIG['reranker'].split('/')[-1]})"),
                   ("total_ms", "Total (retrieval + reranking)")]:
    s = stats(lat[col])
    print(f"\n{label}:")
    print(f"  mean {s['mean']:.1f} ms | median {s['median']:.1f} ms | "
          f"p95 {s['p95']:.1f} ms | range [{s['min']:.1f}, {s['max']:.1f}] ms")

overhead = (lat.rerank_ms.mean() / lat.retrieve_ms.mean()) * 100
print(f"\nReranking adds {lat.rerank_ms.mean():.1f} ms on top of "
      f"{lat.retrieve_ms.mean():.1f} ms retrieval ({overhead:.0f}% relative overhead).")

print(f"""
One-time costs (not per-query):
  BM25 index build     {bm25_build_s:.2f} s  for {len(corpus)} passages
  Dense index build    {index_build_s:.2f} s  for {len(corpus)} passages
  Reranker load        {model_load_s:.2f} s
  Warmup               {warmup_s:.2f} s (first call {first_call_s*1000:.0f} ms)

Suggested paper sentence:
  "On a {str(DEV).upper()}, hybrid retrieval over {len(corpus)} passages took a median of
  {stats(lat['retrieve_ms'])['median']:.0f} ms per query; adding {CONFIG['reranker'].split('/')[-1]}
  reranking of the top-{CONFIG['retrieve_k']} candidates added a median of
  {stats(lat['rerank_ms'])['median']:.0f} ms ({overhead:.0f}% relative overhead), for a total of
  {stats(lat['total_ms'])['median']:.0f} ms per query (p95: {stats(lat['total_ms'])['p95']:.0f} ms)."
""")

print(f"\nAll outputs written under: {OUT_DIR.resolve()}")
for f in ["latency_measurements_tpu.csv"]:
    print("  ", f)

# The original ended with a bare `from google.colab import files`, which raises
# outside Colab and aborted the notebook on the final cell.
try:
    from google.colab import files as colab_files
    for f in ["latency_measurements_tpu.csv"]:
        colab_files.download(out(f))
except ImportError:
    print("(Not in Colab - files are on disk at the path above.)")